In [4]:
import polars as pl
import pandas as pd 
import numpy as np
import os
import pyarrow.dataset as pads
from sklearn.metrics import mean_squared_error,root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import export_text
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import RepeatedKFold,GridSearchCV,RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
def readDataSubset(table_type, data_path = ""):
    if table_type not in ["ball-positions", "ball-events", 'player-positions', 'lineups', "game-info"]:
        print("NO")
        return -1 
    if table_type == 'lineups' or table_type == "game_info":
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type + '.csv'), format = 'csv')
    else:
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type), format = 'csv', partitioning = ['home_team', 'away_team', 'year', 'day'])

In [6]:
# Read Ball events data
ball_events_subset = readDataSubset('ball-events')
ball_events = ball_events_subset.to_table().to_pandas()
ball_events_df = pl.from_pandas(ball_events)

# Read Lineups Data
lineups_subet = readDataSubset('lineups')
lineups = lineups_subet.to_table().to_pandas()
lineups_df = pl.from_pandas(lineups)

In [7]:
ball_events_df.head(5)

game_string,play_per_game,timestamp,player_id,ball_eventcode,home_team,away_team,year,day
str,f64,f64,i64,i64,str,str,str,str
"""y1_d143_ADQ_ANI""",1.0,50804.0,1,1,"""ANI""","""ADQ""","""y1""","""d143"""
"""y1_d143_ADQ_ANI""",1.0,51254.0,2,2,"""ANI""","""ADQ""","""y1""","""d143"""
"""y1_d143_ADQ_ANI""",1.0,51254.0,0,5,"""ANI""","""ADQ""","""y1""","""d143"""
"""y1_d143_ADQ_ANI""",2.0,68204.0,1,1,"""ANI""","""ADQ""","""y1""","""d143"""
"""y1_d143_ADQ_ANI""",2.0,68654.0,2,2,"""ANI""","""ADQ""","""y1""","""d143"""


In [8]:
lineups_df.head(4)

game_string,inning,half_inning,home_team,away_team,at_bat,play_per_game,is_pickoff,pitcher,catcher,first_baseman,second_baseman,third_baseman,shortstop,left_fielder,center_fielder,right_fielder,batter,on_1b,on_2b,on_3b
str,i64,str,str,str,i64,i64,bool,str,str,str,str,str,str,str,str,str,str,str,str,str
"""y1_d061_VKA_PHD""",1,"""Top""","""PHD""","""VKA""",1,1,false,"""PHD-4341""","""PHD-5504""","""PHD-4573""","""PHD-7743""","""PHD-6069""","""PHD-0812""","""PHD-7166""","""PHD-8619""","""PHD-5994""","""VKA-5565""","""NA""","""NA""","""NA"""
"""y1_d061_VKA_PHD""",1,"""Top""","""PHD""","""VKA""",1,2,false,"""PHD-4341""","""PHD-5504""","""PHD-4573""","""PHD-7743""","""PHD-6069""","""PHD-0812""","""PHD-7166""","""PHD-8619""","""PHD-5994""","""VKA-5565""","""NA""","""NA""","""NA"""
"""y1_d061_VKA_PHD""",1,"""Top""","""PHD""","""VKA""",2,3,false,"""PHD-4341""","""PHD-5504""","""PHD-4573""","""PHD-7743""","""PHD-6069""","""PHD-0812""","""PHD-7166""","""PHD-8619""","""PHD-5994""","""VKA-7844""","""NA""","""NA""","""NA"""
"""y1_d061_VKA_PHD""",1,"""Top""","""PHD""","""VKA""",3,4,false,"""PHD-4341""","""PHD-5504""","""PHD-4573""","""PHD-7743""","""PHD-6069""","""PHD-0812""","""PHD-7166""","""PHD-8619""","""PHD-5994""","""VKA-7206""","""NA""","""NA""","""NA"""


## **Stolen Base Model**

- Model based on the odds of a player stealing second base when on first. 
    - features include
        - Player Speed
        - Defensive Positoning
        - Ball location
    - Output would be the should he steal or not (Red light Green Light)
        - Maybe best to use neural network, decision tree, or random forest

**First** Lets build a model only focusing on 2nd baseman positions and the odds to steal based on that. 
- lets first find where a steal happens
- Things we need
    - Player_positioning df
        - game_string
        - Play_per_game when a steal happens
        - timestamp
        - player_id
        - field_x
        - field_y
    - Lineups df
        - game_string
        - play_per_game
        - at_bat == 10
        - second_baseman == 4
        - on_1b == 11
        - on_2b == 12


- Look for squence where playy is put into play but catcher throws to 2nd base

|player_id|event_code|
|-----|-----|
|1 (Pitcher)|1 (pitch)|
|2 (Catcher)|2 (ball acquired)|
|2 (Cather) |3 (Throw)|
|4 (Second BM)| 2 (ball acquired)

In [9]:
# Collect all ball_events where a cather throws the ball
ball_events_ct = ball_events_df.sort(
    ['game_string', 'play_per_game', 'timestamp']
).filter(
    (pl.col("player_id") == 1) &
    (pl.col("ball_eventcode") == 1) &
    (pl.col("player_id").shift(-1) == 2) &
    (pl.col("ball_eventcode").shift(-1) == 2) &
    (pl.col("player_id").shift(-2) == 2) &
    (pl.col("ball_eventcode").shift(-2) == 3)# &
    #(pl.col("player_id").shift(-3) == 4) &
    #(pl.col("ball_eventcode").shift(-3) == 2)
    .over(["game_string", "play_per_game"])
).select(
    ["game_string", "play_per_game", pl.col("timestamp").alias("throw_timestamp")]
)

# Collect all 2nd BM catches with catcher throws before
ball_events_2bm = ball_events_df.sort(
    ['game_string', 'play_per_game', 'timestamp']
).filter(
    (pl.col("player_id").shift(1) == 2) &
    (pl.col("ball_eventcode").shift(1) == 3) #&
   # (pl.col("player_id") == 4) & 
   # (pl.col("ball_eventcode") == 2)
    .over(["game_string", "play_per_game"])
).select(
    ["game_string", "play_per_game", pl.col("timestamp").alias("catch_timestamp")]
)

In [10]:
ball_events_ct.drop_nulls()

game_string,play_per_game,throw_timestamp
str,f64,f64
"""y1_d061_VKA_PHD""",106.0,3.377014e6
"""y1_d061_VKA_PHD""",208.0,6.851814e6
"""y1_d061_VKA_PHD""",213.0,7.066614e6
"""y1_d064_VKA_PHD""",80.0,2.379142e6
"""y1_d064_VKA_PHD""",177.0,5.5483e6
…,…,…
"""y1_d212_PWH_ARN""",201.0,6.578189e6
"""y1_d212_QHX_ANI""",189.0,6.231921e6
"""y1_d213_HNO_ARN""",69.0,2.438464e6


In [11]:
ball_events_ct.shape

(357, 3)

In [12]:
ball_events_ct.head()

game_string,play_per_game,throw_timestamp
str,f64,f64
"""NA""",null,null
"""NA""",null,null
"""NA""",null,null
"""NA""",null,null
"""NA""",null,null


In [13]:
ball_events_2bm.shape

(512, 3)

In [14]:
ball_events_2bm.head()

game_string,play_per_game,catch_timestamp
str,f64,f64
"""NA""",null,null
"""NA""",null,null
"""NA""",null,null
"""NA""",null,null
"""NA""",null,null


In [15]:
# Join df together
steals_df = pl.concat([ball_events_ct, ball_events_2bm], how = "diagonal").group_by(
    ["game_string", "play_per_game"]
).agg(
    # Condonse back into 1 row per game
    pl.col("throw_timestamp").min(),
    pl.col("catch_timestamp").min()
).with_columns(
    # calcualte throw time
    throw_time = ((pl.col("catch_timestamp") - pl.col("throw_timestamp")) / 1000).round(2),
    play_per_game = pl.col("play_per_game").cast(pl.Int64)
).sort(
    # Arraneg by game_string and play_per_game
    ["game_string", "play_per_game"]
)

In [16]:
steals_df.shape

(499, 5)

In [17]:
steals_df.head()

game_string,play_per_game,throw_timestamp,catch_timestamp,throw_time
str,i64,f64,f64,f64
"""NA""",null,null,null,null
"""y1_d061_VKA_PHD""",106,3.377014e6,3.379364e6,2.35
"""y1_d061_VKA_PHD""",208,6.851814e6,6.854164e6,2.35
"""y1_d061_VKA_PHD""",213,7.066614e6,7.069014e6,2.4
"""y1_d062_VKA_PHD""",191,null,5.605734e6,null


In [18]:
steals_df.drop_nulls()

game_string,play_per_game,throw_timestamp,catch_timestamp,throw_time
str,i64,f64,f64,f64
"""y1_d061_VKA_PHD""",106,3.377014e6,3.379364e6,2.35
"""y1_d061_VKA_PHD""",208,6.851814e6,6.854164e6,2.35
"""y1_d061_VKA_PHD""",213,7.066614e6,7.069014e6,2.4
"""y1_d064_VKA_PHD""",80,2.379142e6,2.381419e6,2.28
"""y1_d064_VKA_PHD""",177,5.5483e6,5.550874e6,2.57
…,…,…,…,…
"""y1_d212_PWH_ARN""",201,6.578189e6,6.582314e6,4.12
"""y1_d212_QHX_ANI""",189,6.231921e6,6.234396e6,2.48
"""y1_d213_HNO_ARN""",69,2.438464e6,2.440906e6,2.44


## 2nd Base Man Location

In [19]:
# Make vectors for game_string and play_per_game in throw times
relevant_game_string = steals_df["game_string"]
relevant_ppg = steals_df["play_per_game"]

# read in the player_pos data and filter selected game
player_pos_subset = readDataSubset("player-positions")
player_positions = player_pos_subset.to_table(filter = ((pads.field("game_string").isin(relevant_game_string)) &
                                                        (pads.field("play_per_game").isin(relevant_ppg)) & 
                                                        (pads.field("player_id") == 4))).to_pandas()
player_positions_df = pl.from_pandas(player_positions)

In [20]:
player_positions_df = player_positions_df.with_columns(play_per_game = pl.col("play_per_game").cast(pl.Int64))

In [21]:
player_positions_df.head()

game_string,play_per_game,timestamp,player_id,field_x,field_y,home_team,away_team,year,day
str,i64,f64,i64,f64,f64,str,str,str,str
"""y1_d144_ADQ_ANI""",2,77788.0,4,29.7069,152.3397,"""ANI""","""ADQ""","""y1""","""d144"""
"""y1_d144_ADQ_ANI""",2,77838.0,4,29.7234,152.3361,"""ANI""","""ADQ""","""y1""","""d144"""
"""y1_d144_ADQ_ANI""",2,77888.0,4,29.7396,152.3322,"""ANI""","""ADQ""","""y1""","""d144"""
"""y1_d144_ADQ_ANI""",2,77938.0,4,29.7558,152.3283,"""ANI""","""ADQ""","""y1""","""d144"""
"""y1_d144_ADQ_ANI""",2,77988.0,4,29.7717,152.3241,"""ANI""","""ADQ""","""y1""","""d144"""


In [22]:
player_positions_df.shape

(4974071, 10)

In [23]:
steals_positions = steals_df.join(
    player_positions_df, on = ["game_string", "play_per_game"], how = "left"
).select(
    "game_string", "play_per_game", pl.col("field_x").alias("2nd_baseman_field_x"), pl.col("field_y").alias("2nd_baseman_field_y"), "timestamp", "throw_time"
).sort(
    ["game_string", "play_per_game", "timestamp"]).unique(["play_per_game"])

In [24]:
steals_positions.head(5)

game_string,play_per_game,2nd_baseman_field_x,2nd_baseman_field_y,timestamp,throw_time
str,i64,f64,f64,f64,f64
"""y1_d159_KZR_VAS""",3,-19.5381,152.1735,121873.0,null
"""y1_d067.5_NLG_ANI""",134,26.6193,147.5385,4.359131e6,3.25
"""y1_d153.5_RME_ARN""",137,24.1683,149.2284,4.261787e6,2.4
"""y1_d094_WLI_ARN""",268,-16.4889,143.8389,9.147311e6,2.35
"""y1_d137_EXB_ARN""",265,50.751,143.9901,9.402468e6,3.05


In [25]:
steals_positions.shape

(259, 6)

In [26]:
# FInd the names of second baseman, batter, and runners
postions_names = lineups_df.sort(
    ["game_string", "play_per_game"]
).with_columns(
    pl.col("on_1b").shift(-1).alias("next_on_1b"),
    pl.col("on_2b").shift(-1).alias("next_on_2b"),
    pl.col("at_bat").shift(-1).alias("next_at_bat"),
    pl.col("play_per_game").shift(-1).alias("next_play"),
    play_per_game = pl.col("play_per_game").cast(pl.Int64),
).select(
    ["game_string", "play_per_game", "next_play" , "second_baseman", "at_bat", "next_at_bat", "on_1b", "next_on_1b", "on_2b", "next_on_2b"]
).unique()

In [27]:
postions_names.head(4)

game_string,play_per_game,next_play,second_baseman,at_bat,next_at_bat,on_1b,next_on_1b,on_2b,next_on_2b
str,i64,i64,str,i64,i64,str,str,str,str
"""y1_d149_VNG_VAS""",240,241,"""VAS-1773""",58,58,"""NA""","""NA""","""NA""","""NA"""
"""y1_d110_THF_ARN""",310,311,"""THF-7863""",71,71,"""NA""","""NA""","""NA""","""NA"""
"""y1_d180_YXL_PHD""",222,223,"""YXL-1727""",55,55,"""PHD-8619""","""PHD-8619""","""NA""","""NA"""
"""y1_d143_KVN_VAS""",136,137,"""VAS-1773""",33,34,"""KVN-0052""","""KVN-0052""","""NA""","""NA"""


In [28]:
postions_names.shape

(76455, 10)

 Find all plays where a steal happened
lineups_2nd_steal = postions_names.sort(
    ['game_string', 'play_per_game']
).with_columns(
    pl.col("on_2b").shift(-1).alias("next_on_2b"),
    pl.col("at_bat").shift(-1).alias("next_at_bat"),
    pl.col("play_per_game").shift(-1).alias("next_play"),
    pl.col("play_per_game").shift(1).alias("before_play"),
    pl.lit(True).alias("steal_successful")
).filter(
    (pl.col("on_1b") != "NA") &
    (pl.col("on_1b") == pl.col("next_on_2b")) &
    (pl.col("at_bat") == pl.col("next_at_bat"))
).select(
   ["game_string", "before_play", "play_per_game", pl.col("play_per_game").alias("steal_play"), "next_play", "second_baseman", "at_bat", "next_at_bat", "on_1b", "on_2b", "next_on_2b", "steal_successful"]
)

lineups_2nd_steal.shape

# Find all plays where a steal failed
lineups_2nd_fails = postions_names.sort(
    ["game_string", "play_per_game"]
).with_columns(
    pl.col("on_1b").shift(-1).alias("next_on_1b"),
    pl.col("on_2b").shift(-1).alias("next_on_2b"),
    pl.col("at_bat").shift(-1).alias("next_at_bat"),
    #pl.col("play_per_game").shift(-1).alias("next_play"),
    #pl.col("play_per_game").shift(1).alias("before_play"),
    #pl.col("inning").shift(-1).alias("next_inning"),
    #pl.col("half_inning").shift(-1).alias("next_hafe"),
    pl.lit(False).alias("steal_successful")
).filter(
    (pl.col("on_1b") != "NA") &
    (pl.col("next_on_1b") == "NA") &
    (pl.col("next_on_2b") == "NA") &
    #(pl.col("inning") == pl.col("next_inning")) &
    #(pl.col("half_inning") == pl.col("next_hafe")) &
    (pl.col("at_bat") == pl.col("next_at_bat"))
).select(
   ["game_string", "play_per_game", "second_baseman", "at_bat", "next_at_bat", "on_1b", "next_on_1b", "steal_successful"]
)

In [29]:
joined_df = steals_positions.join(postions_names, on = ["game_string", "play_per_game"], how = "left").select(
    ["game_string", "play_per_game", "next_play" , "throw_time", "2nd_baseman_field_x", "2nd_baseman_field_y", "at_bat", "next_at_bat", "second_baseman", "on_1b", "next_on_1b", "on_2b", "next_on_2b"]
)

In [30]:
joined_df.shape

(263, 13)

In [31]:
joined_df.head(5)

game_string,play_per_game,next_play,throw_time,2nd_baseman_field_x,2nd_baseman_field_y,at_bat,next_at_bat,second_baseman,on_1b,next_on_1b,on_2b,next_on_2b
str,i64,i64,f64,f64,f64,i64,i64,str,str,str,str,str
"""y1_d159_KZR_VAS""",3,4,null,-19.5381,152.1735,1,2,"""VAS-1773""","""NA""","""NA""","""NA""","""NA"""
"""y1_d067.5_NLG_ANI""",134,135,3.25,26.6193,147.5385,33,33,"""NLG-9531""","""ANI-7878""","""NA""","""NA""","""ANI-7878"""
"""y1_d153.5_RME_ARN""",137,138,2.4,24.1683,149.2284,36,36,"""ARN-6658""","""RME-4608""","""NA""","""NA""","""RME-4608"""
"""y1_d094_WLI_ARN""",268,269,2.35,-16.4889,143.8389,65,66,"""WLI-3768""","""ARN-9073""","""NA""","""NA""","""NA"""
"""y1_d137_EXB_ARN""",265,266,3.05,50.751,143.9901,77,77,"""EXB-4050""","""ARN-8132""","""NA""","""NA""","""ARN-8132"""


In [32]:
successful_steal = joined_df.filter(
        (pl.col("on_1b") != "NA") &
        (pl.col("on_1b") == pl.col("next_on_2b")) &
        (pl.col("at_bat") == pl.col("next_at_bat")) &
        (pl.col("on_2b") == "NA")
    ).with_columns(
        (pl.lit(True).alias("steal_successful"))
    )

In [33]:
successful_steal.shape

(47, 14)

In [34]:
successful_steal.head()

game_string,play_per_game,next_play,throw_time,2nd_baseman_field_x,2nd_baseman_field_y,at_bat,next_at_bat,second_baseman,on_1b,next_on_1b,on_2b,next_on_2b,steal_successful
str,i64,i64,f64,f64,f64,i64,i64,str,str,str,str,str,bool
"""y1_d067.5_NLG_ANI""",134,135,3.25,26.6193,147.5385,33,33,"""NLG-9531""","""ANI-7878""","""NA""","""NA""","""ANI-7878""",true
"""y1_d153.5_RME_ARN""",137,138,2.4,24.1683,149.2284,36,36,"""ARN-6658""","""RME-4608""","""NA""","""NA""","""RME-4608""",true
"""y1_d137_EXB_ARN""",265,266,3.05,50.751,143.9901,77,77,"""EXB-4050""","""ARN-8132""","""NA""","""NA""","""ARN-8132""",true
"""y1_d209_QHX_ANI""",399,400,null,27.3393,150.5463,97,97,"""QHX-2389""","""ANI-1431""","""NA""","""NA""","""ANI-1431""",true
"""y1_d115_JPB_ANI""",6,7,2.34,17.3904,146.7333,2,2,"""ANI-6943""","""JPB-2826""","""NA""","""NA""","""JPB-2826""",true


In [35]:
failed_steal = joined_df.filter(
        (pl.col("on_1b") != "NA") &
        (pl.col("on_1b") != pl.col("next_on_2b")) &
        (pl.col("at_bat") == pl.col("next_at_bat")) &
        (pl.col("on_2b") == "NA")
    ).with_columns(
        (pl.lit(False).alias("steal_successful"))
    )

In [36]:
failed_steal.shape

(69, 14)

In [37]:
failed_steal.head()

game_string,play_per_game,next_play,throw_time,2nd_baseman_field_x,2nd_baseman_field_y,at_bat,next_at_bat,second_baseman,on_1b,next_on_1b,on_2b,next_on_2b,steal_successful
str,i64,i64,f64,f64,f64,i64,i64,str,str,str,str,str,bool
"""y1_d105_MSN_VAS""",9,10,2.2,12.5244,150.5193,3,3,"""VAS-1773""","""MSN-2989""","""MSN-2989""","""NA""","""NA""",false
"""y1_d140_QBR_ANI""",15,16,2.5,22.0359,145.8078,3,3,"""ANI-3725""","""QBR-7562""","""QBR-7562""","""NA""","""NA""",false
"""y1_d152_VAS_PHD""",286,287,2.55,33.6813,136.0215,68,68,"""PHD-0812""","""VAS-1238""","""VAS-1238""","""NA""","""NA""",false
"""y1_d095_NFP_PHD""",155,156,3.2,26.349,140.8701,33,33,"""NFP-9038""","""PHD-9663""","""PHD-9663""","""NA""","""NA""",false
"""y1_d075_XPO_PHD""",158,159,2.51,22.3287,141.1899,41,41,"""XPO-9254""","""PHD-8619""","""PHD-8619""","""NA""","""NA""",false


In [38]:
final_df = pl.concat([successful_steal, failed_steal], how = "vertical"
).sort(["game_string", "play_per_game"])

In [39]:
final_df = final_df.with_columns(
    pl.when(pl.col("throw_time").is_null()).then(pl.lit(1)).otherwise(pl.lit(0)).alias("throw_results")
)

In [40]:
final_df.sample(n = 10)

game_string,play_per_game,next_play,throw_time,2nd_baseman_field_x,2nd_baseman_field_y,at_bat,next_at_bat,second_baseman,on_1b,next_on_1b,on_2b,next_on_2b,steal_successful,throw_results
str,i64,i64,f64,f64,f64,i64,i64,str,str,str,str,str,bool,i32
"""y1_d123_IOL_VAS""",98,100,1.95,44.9292,145.9872,26,26,"""VAS-1599""","""IOL-8695""","""IOL-8695""","""NA""","""NA""",false,0
"""y1_d190_PHD_ARN""",160,161,2.48,17.9598,148.9308,35,35,"""ARN-6201""","""PHD-1580""","""NA""","""NA""","""PHD-1580""",true,0
"""y1_d105_MSN_VAS""",9,10,2.2,12.5244,150.5193,3,3,"""VAS-1773""","""MSN-2989""","""MSN-2989""","""NA""","""NA""",false,0
"""y1_d106_MSN_VAS""",305,306,2.45,14.0313,153.5727,74,74,"""VAS-1773""","""MSN-0662""","""MSN-0662""","""NA""","""NA""",false,0
"""y1_d117_BXH_VAS""",50,52,2.05,52.4304,140.7087,14,14,"""VAS-1773""","""BXH-6403""","""BXH-6403""","""NA""","""NA""",false,0
"""y1_d083_RYU_PHD""",123,124,1.45,37.7964,136.1772,38,38,"""PHD-4573""","""RYU-4519""","""NA""","""NA""","""RYU-4519""",true,0
"""y1_d138_HYG_PHD""",11,12,2.11,24.1251,140.9709,3,3,"""PHD-7743""","""HYG-0835""","""NA""","""NA""","""HYG-0835""",true,0
"""y1_d094_NFP_PHD""",139,140,2.2,31.0089,140.901,36,36,"""PHD-7743""","""NFP-7902""","""NFP-7902""","""NA""","""NA""",false,0
"""y1_d072_URD_ARN""",211,213,2.1,33.393,129.5709,50,50,"""ARN-7477""","""URD-1196""","""URD-1196""","""NA""","""NA""",false,0


In [41]:
final_df.shape

(116, 15)

In [42]:
throw_results_count = final_df.filter(pl.col("throw_results") == 1)

throw_results_count.shape

(9, 15)

In [43]:
x_y_df = final_df.select("throw_time", "2nd_baseman_field_x", "2nd_baseman_field_y", "steal_successful", "throw_results")

x_y_df.drop_nulls()
X = x_y_df.drop("steal_successful")
y = x_y_df["steal_successful"]
X = X.to_pandas()
y = y.to_pandas()
y = y.astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = .2, train_size= .8, random_state = 42, stratify = y)

In [44]:
grid_search = {"max_depth": range(1, 16), "max_leaf_nodes": range(2, 21), "min_samples_leaf": range(1, 10)}

decision_tree = DecisionTreeClassifier(random_state = 42)

grid = GridSearchCV(decision_tree, grid_search, cv = 5, scoring = "roc_auc")

grid.fit(X_train, y_train)
print(grid.best_params_)
print(grid.best_score_)
best_tree = grid.best_estimator_

{'max_depth': 3, 'max_leaf_nodes': 4, 'min_samples_leaf': 1}
0.640422077922078


In [45]:
print(export_text(
    best_tree,
    feature_names=list(X.columns)
))

|--- throw_time <= 2.32
|   |--- throw_time <= 1.65
|   |   |--- class: 1
|   |--- throw_time >  1.65
|   |   |--- class: 0
|--- throw_time >  2.32
|   |--- 2nd_baseman_field_y <= 142.70
|   |   |--- class: 0
|   |--- 2nd_baseman_field_y >  142.70
|   |   |--- class: 1



In [46]:
importance = (
    pd.DataFrame({
        "feature": X.columns,
        "importance": best_tree.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

importance

,feature,importance
0,throw_time,0.744803
2,2nd_baseman_field_y,0.255197
1,2nd_baseman_field_x,0.000000
3,throw_results,0.000000


In [47]:
game_filter = lineups_df.filter(
    (pl.col("game_string") == "y1_d184_WIF_ANI"),
    (pl.col("play_per_game").is_in([207, 208]))
    )
game_filter

game_string,inning,half_inning,home_team,away_team,at_bat,play_per_game,is_pickoff,pitcher,catcher,first_baseman,second_baseman,third_baseman,shortstop,left_fielder,center_fielder,right_fielder,batter,on_1b,on_2b,on_3b
str,i64,str,str,str,i64,i64,bool,str,str,str,str,str,str,str,str,str,str,str,str,str
"""y1_d184_WIF_ANI""",7,"""Top""","""ANI""","""WIF""",52,207,false,"""ANI-1492""","""ANI-2268""","""ANI-3725""","""ANI-4963""","""ANI-2218""","""ANI-7149""","""ANI-7878""","""ANI-3166""","""ANI-2830""","""WIF-0714""","""WIF-2990""","""NA""","""NA"""
"""y1_d184_WIF_ANI""",7,"""Top""","""ANI""","""WIF""",52,208,false,"""ANI-1492""","""ANI-2268""","""ANI-3725""","""ANI-4963""","""ANI-2218""","""ANI-7149""","""ANI-7878""","""ANI-3166""","""ANI-2830""","""WIF-0714""","""WIF-2990""","""NA""","""NA"""
